# 06 · Gegevens opvragen via een API

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De weegschaal en de voedercomputer sturen hun metingen via MQTT naar jouw server. In notebook 05 heb je beide meetstromen verzameld in één database. Vanuit een kantoorcomputer wil je die gegevens nu gericht opvragen, zodat je ze straks samen kunt analyseren. Daarvoor breid je de webdienst op je server uit met een **API** die meetgegevens kan teruggeven.

Je vraagt eerst enkele losse records op, kiest daarna zelf een selectie en vult tot slot een lokale analysedatabase. Daarna controleer je of die kopie precies dezelfde metingen bevat als de database op je server.

**Nodig:** je Codespace en eigen branch, je Droplet uit notebook 03 en een afgerond notebook 05. Houd je **`SSH beheer-wachtwoord`** en **`API-Droplet wachtwoord`** bij de hand. Je werkt in dit notebook met beide gegevensbronnen, ongeacht je rol uit notebook 01.

Voer de codecellen in volgorde uit. Werk bij de serverstappen eerst de opdrachten in de SSH-terminal af voordat je verdergaat met de controle in het notebook.

</div>

## 1. De werkplek en ontvangen gegevens controleren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

In deze oefening speelt Codespaces ook de rol van de kantoorcomputer die gegevens opvraagt en een eigen analysebestand bewaart. Je hoeft daarvoor geen aparte Codespace te openen, omdat elke rol zijn eigen bestanden en programma's gebruikt.

De volgende cel haalt je instellingen, serveradres en certificaat op uit de vorige notebooks en controleert of het inleverbestand van notebook 05 bij jouw branch hoort.

</div>

In [1]:
_DOCENT_BRANCH_OVERRIDE = "student/proefdocent"

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
from contextlib import closing
import ipaddress
import json
import re
import sqlite3
import subprocess
import sys
import pandas as pd

r = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("Open notebook 06 vanuit de practicumrepository in Codespaces.")
hoofdmap = Path(r.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch) and "_DOCENT_BRANCH_OVERRIDE" in globals():
    branch = _DOCENT_BRANCH_OVERRIDE
instellingen = json.loads((hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json").read_text(encoding="utf-8"))
deelnemer, rol = instellingen["deelnemerscode"], instellingen["rol"]
if branch != "student/" + deelnemer or instellingen.get("branch") != branch or rol not in ("gewichten", "voederopname"):
    raise ValueError("Controleer je eigen branch en instellingen uit notebook 01.")
werkmap = hoofdmap / "gegevens/werk"
server = json.loads((werkmap / "server.json").read_text(encoding="utf-8"))
if server.get("deelnemerscode") != deelnemer:
    raise ValueError("De serverinstellingen horen bij een andere deelnemer.")
serveradres = str(ipaddress.IPv4Address(server["ipv4"]))
ca = werkmap / "ca.crt"
if not ca.is_file():
    raise FileNotFoundError("Haal je openbare ca.crt opnieuw op volgens notebook 03.")
ontvangst05 = json.loads((hoofdmap / "gegevens/inlevering/05_ontvangstcontrole.json").read_text(encoding="utf-8"))
if (ontvangst05.get("deelnemerscode") != deelnemer or ontvangst05.get("branch") != branch
    or ontvangst05.get("volledigheid") != "geslaagd"
    or set(ontvangst05.get("bronnen", {})) != {"asserva", "schauer"}):
    raise ValueError("Rond de inhoudscontrole en inlevering van notebook 05 af.")
sys.path[:0] = [str(hoofdmap / "programmatuur"), str(hoofdmap / "server")]
from api_ophalen import ApiClient, werk_analyse_bij
print("Deelnemer:", deelnemer, "| Rol:", rol)
print("Beide bronnen gecontroleerd in notebook 05:", {b: v["aantal"] for b, v in ontvangst05["bronnen"].items()})

Deelnemer: proefdocent | Rol: gewichten
Beide bronnen gecontroleerd in notebook 05: {'asserva': 80028, 'schauer': 146649}


| Bestand of programma | Computer | Functie |
|---|---|---|
| `asserva.db` of `schauer.db` | Toestelcomputer in Codespaces | Brongegevens publiceren |
| Mosquitto en `iot-ontvanger` | Eigen Droplet | Berichten verdelen en opslaan |
| `verzamelde_metingen.db` | Eigen Droplet | Ontvangen gegevens van beide toestellen bewaren |
| FastAPI achter Nginx | Eigen Droplet | Gerichte HTTPS-aanvragen beantwoorden |
| `analyse_06.db` | Kantoorcomputer in Codespaces | Opgevraagde gegevens voor berekeningen bewaren |

![Gegevens van toestelcomputers via MQTT naar de server en via HTTPS naar de kantoorcomputer](../../afbeeldingen/api_gegevensstroom.svg)

*Figuur 1. De API leest de verzameldatabase. De kantoorcomputer ontvangt een antwoord op een aanvraag.*

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 1.** Waarom heeft de kantoorcomputer nog een API nodig als de metingen al via MQTT zijn verstuurd?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 1):**

*Voeg hier je antwoord in. Tip: vergelijk het ontvangen van nieuwe berichten met het later opvragen van opgeslagen gegevens.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 1**

MQTT stuurt nieuwe berichten alleen door naar programma's die op dat moment verbonden zijn. De ontvanger op de server heeft die metingen bewaard in een database. Via de API kan de kantoorcomputer die opgeslagen gegevens op elk later moment en per selectie opvragen, zonder dat hij eerder live via MQTT hoefde mee te luisteren.

</div>

## 2. Wat gebeurt er bij een API-aanvraag?

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een **API** legt vast hoe programma's gegevens of functies bij elkaar kunnen opvragen. Hier gebruik je een web-API: Python stuurt een aanvraag naar een webadres op je server en krijgt een antwoord terug. **HTTP** bepaalt de vorm van die aanvraag en dat antwoord, en **HTTPS** voegt daar versleuteling en controle van de serveridentiteit aan toe.

Met **GET** vraag je gegevens op. De **URL** wijst naar je server en het gekozen pad, en met **parameters** beperk je de selectie. Het antwoord bevat een HTTP-statuscode en de opgevraagde gegevens in JSON. Je kunt alleen selecties opvragen die de API ondersteunt.

In het voorbeeld hieronder vraagt de kantoorcomputer maximaal vijf gewichtsrecords van hok 1 op. Dit is een leesvoorbeeld, nog geen opdracht om uit te voeren.

</div>

```text
GET https://<jouw-serveradres>/metingen?meettype=gewichten&hok=1&limiet=5
```

| Onderdeel | Betekenis |
|---|---|
| `https://` | Versleutelde webverbinding |
| `<jouw-serveradres>` | Je eigen Droplet |
| `/metingen` | Aanspreekpunt voor meetrecords |
| `?meettype=gewichten` | Begin van de parameters, kies het meettype |
| `&hok=1` | Beperk de selectie tot hokcode `1` |
| `&limiet=5` | Vraag maximaal vijf records in één antwoord |
| HTTP `200` | De aanvraag is gelukt, ook wanneer de selectie leeg is |

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Uitleg</strong><br>Op de server verwerkt <strong>FastAPI</strong> je aanvraag en <strong>Uvicorn</strong> voert dat Python-programma uit. <strong>Nginx</strong> vangt het HTTPS-verkeer op en stuurt het intern door. Die onderdelen heb je al geïnstalleerd in notebook 03. Dit notebook werkt nu als client.</div>

## 3. De meetgegevens-API op je server installeren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** eerst in Codespaces (Notebook 06), daarna in Codespaces (Terminal)

Je breidt de bestaande dienst `iot-status` uit. Het serveradres, het certificaat en je **`API-Droplet wachtwoord`** blijven gelijk. De API krijgt alleen leesrechten op de verzameldatabase, zodat de ontvanger intussen rustig nieuwe MQTT-metingen kan blijven opslaan.

**Stap 1.** Voer de cel hieronder uit en kopieer de geprinte `scp`-opdracht naar een terminal in de hoofdmap van de repository. Zo kopieer je het API-programma en de dienstinstelling naar de map van `beheer` op je server. Vul bij de vraag om een wachtwoord je **`SSH beheer-wachtwoord`** in (je ziet daarbij geen tekens verschijnen).

</div>

In [3]:
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 server/webdienst.py server/configuratie/api-uitbreiding.conf beheer@{serveradres}:~/")

scp -o HostKeyAlgorithms=ssh-ed25519 server/webdienst.py server/configuratie/api-uitbreiding.conf beheer@127.0.0.1:~/


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Na een geslaagde kopie zie je beide bestandsnamen en een voortgang die eindigt op `100%`.

**Stap 2.** Maak vanuit dezelfde Codespaces-terminal een SSH-verbinding. De volgende cel drukt de opdracht met je eigen adres af. Gebruik opnieuw je **`SSH beheer-wachtwoord`**.

</div>

In [4]:
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal)

**Stap 3.** Controleer eerst of de verzameldatabase op de server staat. De opdracht toont de bestandsnaam, grootte en eigenaar. Ontbreekt dit bestand, werk dan eerst notebook 05 af. `sudo` kan om je **`SSH beheer-wachtwoord`** vragen.

</div>

```bash
sudo ls -lh /var/lib/iot/verzamelde_metingen.db
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 4.** Plaats het API-programma naast de andere serverprogramma's. De eigenaar `root` mag het programma wijzigen. De dienstgebruiker `iot` kan het lezen. Deze opdracht verandert de database en het API-wachtwoord niet.

</div>

```bash
sudo install -o root -g iot -m 0640 ~/webdienst.py /opt/iot/programmatuur/webdienst.py
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 5.** Voeg de extra instelling voor de dienst toe. Een bestand in `iot-status.service.d` vult de bestaande service aan en vervangt de startopdracht door het volledige API-programma. De API luistert net als voorheen alleen intern op `127.0.0.1:8000`, dus je hoeft geen extra firewallpoort te openen.

</div>

```bash
sudo mkdir -p /etc/systemd/system/iot-status.service.d
sudo install -o root -g root -m 0644 ~/api-uitbreiding.conf /etc/systemd/system/iot-status.service.d/api-uitbreiding.conf
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 6.** Laat systemd de gewijzigde dienstinstelling lezen en herstart alleen de webdienst. Controleer daarna de toestand.

</div>

```bash
sudo systemctl daemon-reload
sudo systemctl restart iot-status
sudo systemctl is-active iot-status
```

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>De laatste opdracht toont <code>active</code>. Dit bewijst dat het proces draait. De volgende sectie controleert ook de HTTPS-verbinding, je aanmelding en de database-inhoud. Bij <code>failed</code> gebruik je de foutentabel onderaan dit notebook.</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip</strong><br>Bij opnieuw doorlopen mag je dezelfde bestanden opnieuw installeren en de dienst herstarten. De database en wachtwoorden blijven behouden. De API start na een serverherstart automatisch via de bestaande dienst. Mosquitto blijft volgens notebook 03 handmatig te starten.</div>

## 4. Je eerste beveiligde aanvraag uitvoeren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Je gebruikt opnieuw `ca.crt` om de identiteit van je server te controleren en meldt je aan met gebruiker `eigenaar` en je **`API-Droplet wachtwoord`** (dit is een ander wachtwoord dan die van de MQTT-broker).

Wanneer je de cel uitvoert, verschijnt **helemaal bovenaan het scherm** een klein invoerbalkje voor je **`API-Droplet wachtwoord`**. Vul het in en druk op **Enter**. Het wachtwoord wordt alleen tijdens deze sessie gebruikt en nergens in het notebook of Git opgeslagen.

De meegeleverde `ApiClient` regelt automatisch de certificaatcontrole, het inloggen en een korte pauze tussen aanvragen, zodat je server niet overbelast raakt. Wat je precies opvraagt blijft zichtbaar in de codecellen.

</div>

In [5]:
if "api" in globals():
    api.sluiten()
api = ApiClient(f"https://{serveradres}", ca, getpass("API-Droplet wachtwoord: "))
antwoord = api.antwoord("/gezondheid")
print("HTTP-status:", antwoord.status_code)
if antwoord.status_code != 200:
    raise RuntimeError("Controleer de foutentabel voordat je verdergaat.")
print(json.dumps(antwoord.json(), ensure_ascii=False, indent=2))
assert antwoord.json().get("api_versie") == 1, "Installeer eerst de uitbreiding uit sectie 3."

HTTP-status: 200
{
  "status": "bereikbaar",
  "dienst": "practicum",
  "api_versie": 1
}


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>Je verwacht HTTP <code>200</code>, <code>status: bereikbaar</code> en <code>api_versie: 1</code>. HTTP <code>401</code> betekent dat de server bereikbaar is maar de aanmelding niet klopt. Ontbreekt <code>api_versie</code>, dan draait nog de oorspronkelijke statusdienst.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Vraag nu `/metadata` op. Dit antwoord beschrijft welke bronnen, eenheden en hokken beschikbaar zijn. De aantallen en inhoudsvingerafdrukken moeten overeenkomen met de controle uit notebook 05.

</div>

In [6]:
metadata = api.ophalen("/metadata")
print(json.dumps(metadata, ensure_ascii=False, indent=2))
if metadata["bronnen"] != ontvangst05["bronnen"]:
    raise ValueError("De API-inhoud verschilt van je controle uit notebook 05. Controleer of je de juiste server gebruikt en of er nieuwe data zijn toegevoegd.")
print("Beide bronnen komen overeen met notebook 05.")

{
  "api_versie": 1,
  "grens": 226677,
  "bronnen": {
    "asserva": {
      "aantal": 80028,
      "sha256": "894618c5a3275a33c2fc75841814a4d18948351c70afc8cf6c1a65eccb7f0a58"
    },
    "schauer": {
      "aantal": 146649,
      "sha256": "7d8c99caf64a6c128074b81dcf0de003d10f54887fe39a3489ebf0d52d85b249"
    }
  },
  "hokken": [
    "1",
    "10",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9"
  ],
  "eenheden": {
    "gewichten": "kg",
    "voederopname": "g"
  },
  "datumveld": "bronrecord.DATE",
  "tijdzone": "onbevestigd; vermoedelijk Europe/Brussels",
  "selectie": "Alle ontvangen bronrecords, inclusief kwaliteitsvlaggen.",
  "paginering": "Volg volgende_na met dezelfde grens en filters. Volgorde is opslagvolgorde, niet meettijd."
}
Beide bronnen komen overeen met notebook 05.


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een **inhoudsvingerafdruk** is een controlecode die berekend wordt over alle recordidentificaties en bijbehorende meetwaarden. Alleen het aantal records vergelijken is niet genoeg, omdat twee datasets even groot kunnen zijn maar toch andere inhoud kunnen hebben. Dit controleert nog niet of elke meting later geschikt is voor een groeiberekening.

</div>

## 5. De API-documentatie en een meetrecord lezen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Een API heeft documentatie nodig, zodat de client weet welke aanvragen mogelijk zijn. FastAPI beschrijft de beschikbare paden en parameters in het gestandaardiseerde **OpenAPI-formaat**. Je vraagt die documentatie op via dezelfde beveiligde verbinding. De tabel hieronder wordt uit dat antwoord opgebouwd.

</div>

In [7]:
documentatie = api.ophalen("/openapi.json")
parameterschema = documentatie["paths"]["/metingen"]["get"]["parameters"]
display(pd.DataFrame([{"parameter": p["name"], "verplicht": p.get("required", False),
                       "schema": json.dumps(p["schema"], ensure_ascii=False)} for p in parameterschema]))

,parameter,verplicht,schema
0,meettype,False,"{""anyOf"": [{""enum"": [""gewichten"", ""voederopnam..."
1,hok,False,"{""anyOf"": [{""type"": ""string"", ""maxLength"": 100..."
2,dier_id,False,"{""anyOf"": [{""type"": ""string"", ""maxLength"": 100..."
3,vanaf,False,"{""anyOf"": [{""type"": ""string"", ""format"": ""date""..."
4,tot,False,"{""anyOf"": [{""type"": ""string"", ""format"": ""date""..."
5,limiet,False,"{""type"": ""integer"", ""maximum"": 1000, ""minimum""..."
6,na,False,"{""type"": ""integer"", ""maximum"": 922337203685477..."
7,grens,False,"{""anyOf"": [{""type"": ""integer"", ""maximum"": 9223..."


| Pad | Gebruik |
|---|---|
| `/gezondheid` | Bereikbaarheid en API-versie |
| `/metadata` | Bronnen, aantallen, hokken, eenheden en inhoudscontrole |
| `/metingen` | Meetrecords, eventueel gefilterd |
| `/openapi.json` | Beschrijving van de beschikbare API-aanvragen |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Vraag nu één record op van je eigen meettype. De JSON-structuur rond het antwoord geeft algemene informatie over de aanvraag, terwijl de lijst `metingen` de echte records bevat. Met `[0]` selecteer je het eerste record uit die lijst.

</div>

In [8]:
eerste_pagina = api.ophalen("/metingen", {"meettype": rol, "limiet": 1})
print(json.dumps(eerste_pagina, ensure_ascii=False, indent=2))
if not eerste_pagina["metingen"]:
    raise ValueError("Je eigen meettype bevat geen records. Controleer notebook 05.")
eerste = eerste_pagina["metingen"][0]
print("Record:", eerste["record_id"])
print("Meetwaarde:", eerste["waarde"], eerste["eenheid"])
print("Hokmetadata:", eerste["hokmetadata"])

{
  "api_versie": 1,
  "grens": 226677,
  "aantal": 1,
  "volgende_na": 1,
  "metingen": [
    {
      "bron": "asserva",
      "bronrecord": {
        "DATAORIGIN": "allocation",
        "DATE": "2023-08-24",
        "DTMARKER": 0.5,
        "DTMARKER_INT": 0.0,
        "EID": "89728010",
        "ERROR": "no error",
        "FILTER": "0",
        "FLAG_VBW_hampel": "0",
        "FLAG_VBW_matern1": "0",
        "FLAG_VBW_matern2": "0",
        "FLAG_VBW_polyn": "0",
        "GENDER": "0",
        "PEN": "3",
        "PROCESSTIME": 0.0,
        "ROW_ID": "897280101692878400000000000",
        "TIMESTAMPstart": "20230824120000",
        "TIMESTAMPstop": "20230824120001",
        "TIMEstart": "12:00:00",
        "TIMEstop": "12:00:01",
        "TREAT": "C",
        "TRIALDAY": 0.5,
        "VBW": 6.68
      },
      "dier_id": "89728010",
      "eenheid": "kg",
      "hok": "3",
      "hokmetadata": {
        "afdeling": 1,
        "gebouw": "M5",
        "hok": "3",
        "oppervlakte

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 2.** Welke eenheid heeft jouw eerste record, en waar vind je de hokoppervlakte terug?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 2):**

*Voeg hier je antwoord in. Tip: gebruik de velden eenheid en hokmetadata in jouw uitvoer.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 2**

Een gewichtsrecord gebruikt de eenheid `kg` en een voederopnamerecord gebruikt `g`. De hokoppervlakte staat onder `hokmetadata` in het veld `oppervlakte_m2` (in deze oefening `8`). De API geeft dus dezelfde broninhoud en hokgegevens terug die via MQTT zijn opgeslagen.

</div>

## 6. Zelf een gerichte selectie opvragen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Met een gerichte selectie vraag je bijvoorbeeld alleen de metingen van één hok over een paar dagen op. De API ondersteunt `meettype`, `hok`, `dier_id`, `vanaf` en `tot`. De datums worden vergeleken met het bronveld `DATE`: `vanaf` telt wel mee in de selectie, `tot` niet meer. De periode van `2023-08-24` tot `2023-08-27` levert dus 24, 25 en 26 augustus op.

**Doe-opdracht.** Kies een hok uit `metadata["hokken"]` en vraag voor die periode vijf voederopnamerecords op, ook als jouw eigen rol `gewichten` is. Gebruik de parameternamen uit sectie 2. Een hoknummer geef je door als tekst, bijvoorbeeld `"1"`.

Vul hieronder de dictionary `selectie` aan en haal de records op met `api.ophalen`. De controlecel eronder toont daarna de belangrijkste velden.

</div>

In [9]:
# SCHRIJF HIER JE CODE:
# 1. Maak selectie met meettype, hok, vanaf, tot en limiet.
# 2. Kies een hokcode uit metadata["hokken"].
# 3. Bewaar api.ophalen("/metingen", selectie) als geselecteerd.

In [10]:
selectie = {"meettype": "voederopname", "hok": metadata["hokken"][0],
            "vanaf": "2023-08-24", "tot": "2023-08-27", "limiet": 5}
geselecteerd = api.ophalen("/metingen", selectie)

In [11]:
assert selectie["meettype"] == "voederopname" and selectie["limiet"] == 5
assert selectie["hok"] in metadata["hokken"]
assert selectie["vanaf"] == "2023-08-24" and selectie["tot"] == "2023-08-27"
assert geselecteerd["metingen"], "Deze selectie is leeg. Kies een ander hok."
for b in geselecteerd["metingen"]:
    assert b["hok"] == selectie["hok"] and b["meettype"] == selectie["meettype"]
    assert selectie["vanaf"] <= b["bronrecord"]["DATE"] < selectie["tot"]
display(pd.DataFrame([{"record_id": b["record_id"], "hok": b["hok"],
                       "datum": b["bronrecord"]["DATE"], "voederopname_g": b["waarde"]}
                      for b in geselecteerd["metingen"]]))

,record_id,hok,datum,voederopname_g
0,897280261692879735000000000,1,2023-08-24,0.0
1,897280261692888293000000000,1,2023-08-24,1.0
2,897280261692888311000000000,1,2023-08-24,0.0
3,897280261692895329000000000,1,2023-08-24,0.0
4,897280261692897467000000000,1,2023-08-24,0.0


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Interpretatie</strong><br>Deze vijf records vormen maar een kleine voorlopige pagina en geen dagsom of gemiddelde. De volgorde is de opslagvolgorde op de server. Zo'n klein voorbeeld is handig om het formaat te bekijken, maar voor een echte analyse moet je alle pagina's ophalen.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 3.** Waarom kun je met deze vijf records nog niet de gemiddelde voederopname per dag voor het hok berekenen?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 3):**

*Voeg hier je antwoord in. Tip: denk aan limiet, individuele bezoeken en het aantal dieren.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 3**

Door `limiet=5` krijg je maar vijf losse bezoekrecords terug en mis je alle andere bezoeken van die dagen. Bovendien weet je met vijf willekeurige bezoeken nog niet welke dieren geldig zijn en hoe je de gegevens per dag en per dier of hok moet optellen.

</div>

## 7. Een lege selectie en een fout onderscheiden

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Een geldige vraag waarvoor geen metingen bestaan, heeft een heel andere betekenis dan een ongeldige aanvraag. Je test beide gevallen nu apart. Eerst vraag je een hok op dat niet bestaat, en daarna stuur je een ongeldige parameter mee (`limiet=0`, terwijl minstens `1` vereist is).

</div>

In [12]:
leeg = api.antwoord("/metingen", {"hok": "bestaat-niet"})
fout = api.antwoord("/metingen", {"limiet": 0})
print("Lege selectie:", leeg.status_code, leeg.json())
print("Ongeldige aanvraag:", fout.status_code)
print(json.dumps(fout.json(), ensure_ascii=False, indent=2))
assert leeg.status_code == 200 and leeg.json()["metingen"] == []
assert fout.status_code == 422

Lege selectie: 200 {'api_versie': 1, 'grens': 226677, 'aantal': 0, 'volgende_na': None, 'metingen': []}
Ongeldige aanvraag: 422
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "query",
        "limiet"
      ],
      "msg": "Input should be greater than or equal to 1",
      "input": "0",
      "ctx": {
        "ge": 1
      }
    }
  ]
}


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 4.** Welke handeling past bij HTTP 200 met een lege lijst, en welke bij HTTP 422?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 4):**

*Voeg hier je antwoord in. Tip: vergelijk een ontbrekende selectie met een verkeerde parameter.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 4**

Bij HTTP 200 met een lege lijst was de aanvraag zelf geldig: je controleert dan of het gevraagde hoknummer of de datumkeuze klopt. Bij HTTP 422 was een parameter ongeldig (hier `limiet=0`), dus pas je die parameter aan en stuur je de aanvraag opnieuw. In geen van beide gevallen hoef je iets op de server opnieuw te installeren.

</div>

## 8. Beide bronnen in een analysedatabase bewaren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

De eerste vier weken tellen samen 226.677 bronrecords. Die zijn te groot voor één aanvraag, dus haal je ze op in pagina's van maximaal duizend records. Het veld `volgende_na` wijst steeds naar het begin van de volgende pagina, terwijl `grens` de bovenkant van de momentopname vastlegt. Eventuele nieuwere metingen gaan dan bij een volgende ophaalronde mee.

De functie `werk_analyse_bij` haalt automatisch alle pagina's op, controleert de inhoud en slaat alles op in **`gegevens/werk/analyse_06.db`**. Bij de eerste keer maakt ze dat bestand aan, en bij een herhaling voegt ze alleen ontbrekende metingen toe. Gaat er onderweg iets mis, dan wordt die halve ronde teruggedraaid.

Je eigen `asserva.db` of `schauer.db` blijft het bronbestand om te publiceren en mag je hier nooit als analysedatabase opgeven. De cel hieronder controleert of je start met een nieuw of een al bestaand analysebestand.

</div>

In [13]:
analysepad = werkmap / "analyse_06.db"
print("Analysebestand:", analysepad.relative_to(hoofdmap))
print("Bestaand analysebestand wordt bijgewerkt." if analysepad.exists() else "Er wordt een nieuwe, lege analysedatabase aangemaakt.")
eerste_import = werk_analyse_bij(api.ophalen, analysepad)
if eerste_import["bronnen"] != ontvangst05["bronnen"]:
    raise ValueError("Deze gegevens horen niet meer bij de gecontroleerde beginsituatie uit notebook 05.")
print(json.dumps(eerste_import, ensure_ascii=False, indent=2))

Analysebestand: gegevens/werk/analyse_06.db
Er wordt een nieuwe, lege analysedatabase aangemaakt.


25,000 records gecontroleerd


50,000 records gecontroleerd


75,000 records gecontroleerd


100,000 records gecontroleerd


125,000 records gecontroleerd


150,000 records gecontroleerd


175,000 records gecontroleerd


200,000 records gecontroleerd


225,000 records gecontroleerd


Klaar: 226,677 nieuwe records, 0 ongewijzigde records.
{
  "gecontroleerd_op_utc": "2026-11-25T09:05:20.456592+00:00",
  "nieuw": 226677,
  "ongewijzigd": 0,
  "bronnen": {
    "asserva": {
      "aantal": 80028,
      "sha256": "894618c5a3275a33c2fc75841814a4d18948351c70afc8cf6c1a65eccb7f0a58"
    },
    "schauer": {
      "aantal": 146649,
      "sha256": "7d8c99caf64a6c128074b81dcf0de003d10f54887fe39a3489ebf0d52d85b249"
    }
  },
  "grens": 226677
}


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>Bij een nieuw bestand worden 80.028 gewichtsrecords en 146.649 voederopnamerecords toegevoegd. Bestond je analysedatabase al, dan kan de teller nul nieuwe records tonen. In beide gevallen moeten de aantallen en inhoudsvingerafdrukken gelijk zijn aan notebook 05. De voortgang verschijnt om de 25 pagina's.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Bekijk nu met SQL wat er lokaal in je analysedatabase staat. Deze query leest alleen je lokale bestand in Codespaces en stuurt niets naar de API. Aan de eerste en laatste datum zie je meteen welke periode nu beschikbaar is.

</div>

In [14]:
with closing(sqlite3.connect(analysepad.as_uri() + "?mode=ro", uri=True)) as db:
    samenvatting = pd.read_sql_query("""
        SELECT bron, meettype, eenheid, COUNT(*) AS records,
               COUNT(DISTINCT hok) AS hokken, MIN(datum) AS eerste_dag, MAX(datum) AS laatste_dag
        FROM metingen GROUP BY bron, meettype, eenheid ORDER BY bron
    """, db)
display(samenvatting)
assert set(samenvatting["bron"]) == {"asserva", "schauer"}

,bron,meettype,eenheid,records,hokken,eerste_dag,laatste_dag
0,asserva,gewichten,kg,80028,10,2023-08-24,2023-09-20
1,schauer,voederopname,g,146649,10,2023-08-24,2023-09-20


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De analysedatabase bewaart ook de volledige JSON-inhoud en de kwaliteitsvlaggen van ieder record. De API filtert nog niet op bruikbare dieren of geldige groei: die keuzes maak je pas bij de analyse zelf. De getallen hier zijn dus aantallen ruwe bronrecords, nog geen bruikbare bezoeken.

</div>

## 9. Herhaald ophalen en verderwerken testen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

In de praktijk haal je gegevens vaker op dan één keer. De combinatie van `bron` en `record_id` zorgt ervoor dat bestaande metingen herkend worden en niet dubbel worden opgeslagen. Draai de volledige import nu een tweede keer om te testen of het aantal nieuwe records netjes op nul blijft staan.

</div>

In [15]:
herhaalde_import = werk_analyse_bij(api.ophalen, analysepad)
assert herhaalde_import["nieuw"] == 0, "Er zijn nieuwe bronrecords aangekomen. Controleer dit voordat je de herhaalproef beoordeelt."
assert herhaalde_import["bronnen"] == eerste_import["bronnen"]
print("Herhaling voegt geen dubbele records toe.")

25,000 records gecontroleerd


50,000 records gecontroleerd


75,000 records gecontroleerd


100,000 records gecontroleerd


125,000 records gecontroleerd


150,000 records gecontroleerd


175,000 records gecontroleerd


200,000 records gecontroleerd


225,000 records gecontroleerd


Klaar: 0 nieuwe records, 226,677 ongewijzigde records.
Herhaling voegt geen dubbele records toe.


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 5.** Wat gebeurt er met je lokale analysedatabase als je Codespace stopt, of als de API even niet bereikbaar is?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 5):**

*Voeg hier je antwoord in. Tip: onderscheid bestaande bestanden, nieuwe aanvragen en een volledig nieuwe Codespace.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 5**

Als je dezelfde Codespace stopt en later weer hervat, blijft `gegevens/werk/analyse_06.db` gewoon bewaard. Je kunt de al opgehaalde gegevens dus blijven analyseren, ook als de API tijdelijk onbereikbaar is. Alleen voor het ophalen van nieuwe metingen moet de API bereikbaar zijn. Start je ooit een volledig nieuwe Codespace, dan staat dat werkbestand er niet in en haal je het opnieuw op via de API.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip</strong><br>Hervat later dezelfde Codespace en voer de cellen opnieuw uit. Het wachtwoord wordt opnieuw gevraagd, maar het analysebestand blijft behouden. Bij een onderbroken import herhaal je de volledige importcel. Je hoeft daarvoor geen databases te verwijderen.</div>

## 10. Resultaat bewaren en afsluiten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

De volgende cel maakt automatisch een compact controlebestand aan op basis van je aanvragen, selectie, foutproef en importcontroles. Je hoeft geen apart verslag te schrijven: je antwoorden bij de vragen blijven gewoon in dit notebook staan. Het controlebestand bevat geen wachtwoord of serveradres.

</div>

In [16]:
assert leeg.status_code == 200 and fout.status_code == 422
assert herhaalde_import["nieuw"] == 0
assert eerste_import["bronnen"] == herhaalde_import["bronnen"] == ontvangst05["bronnen"]
verslag = {
    "deelnemerscode": deelnemer, "branch": branch, "rol": rol,
    "getest_op_utc": datetime.now(timezone.utc).isoformat(), "api_versie": metadata["api_versie"],
    "https_en_aanmelding": "geslaagd", "selectie": selectie,
    "selectie_aantal": geselecteerd["aantal"], "lege_selectie_status": leeg.status_code,
    "ongeldige_aanvraag_status": fout.status_code,
    "eerste_import": eerste_import, "herhaalde_import": herhaalde_import,
    "inhoud_gelijk_aan_05": True, "analysebestand": "gegevens/werk/analyse_06.db",
}
inleverpad = hoofdmap / "gegevens/inlevering/06_apicontrole.json"
inleverpad.write_text(json.dumps(verslag, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
api.sluiten()
print("Bewaard:", inleverpad.relative_to(hoofdmap))
print("De API-sessie is gesloten. Voor nieuwe aanvragen voer je de aanmeldcel opnieuw uit.")

Bewaard: gegevens/inlevering/06_apicontrole.json
De API-sessie is gesloten. Voor nieuwe aanvragen voer je de aanmeldcel opnieuw uit.


<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Stop je Codespace na het practicum</strong><br>Bewaar je werk, commit en push voordat je de Codespace stopt. De Droplet heeft een afzonderlijke facturering. Laat die beschikbaar voor de volgende oefening en verwijder hem alleen volgens de afspraak met de docent.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Source Control) en op de GitHub-website

1. Controleer of je antwoorden zijn ingevuld en kies **File → Save All**.
2. Selecteer dit notebook en `gegevens/inlevering/06_apicontrole.json` in **Source Control**. Wil je je serveradres niet in het notebook bewaren, wis dan eventuele tijdelijke uitvoer voordat je opslaat.
3. Commit met bijvoorbeeld **API-aanvragen en analysedatabase gecontroleerd**, kies **Push** en controleer op GitHub of beide bestanden op je branch staan.
4. `analyse_06.db` blijft in `gegevens/werk/` en gaat niet mee naar GitHub. Als je ooit je Codespace zou verwijderen, kun je dat bestand vooraf downloaden (zodra de importcel klaar is, want die sluit de database netjes af).
5. Sluit de SSH-terminal met `exit` en stop je Codespace via [github.com/codespaces](https://github.com/codespaces).

In notebook 07 gebruik je `analyse_06.db` om groei, voederopname en voederconversie per hok te berekenen. Bewaar dus bij voorkeur dezelfde Codespace en dit analysebestand.

</div>

## Hulp bij veelvoorkomende problemen

| Probleem | Controle en herstel |
|---|---|
| Controlebestand 05 ontbreekt | Werk notebook 05 en de inlevercel eerst volledig af. Maak geen controlebestand met de hand aan. |
| ImportError voor api_ophalen of webdienst | Controleer of alle bestanden van notebook 06 op je branch staan en correct naar de server gekopieerd zijn. |
| Certificaatcontrole mislukt | Controleer je serveradres, de geldigheid en het via SSH opgehaalde `ca.crt`. Schakel certificaatcontrole niet uit. |
| HTTP 401 | Voer de aanmeldcel opnieuw uit met je **`API-Droplet wachtwoord`**. |
| HTTP 404 voor /metingen | Controleer of `api-uitbreiding.conf` goed geplaatst is en herstart `iot-status`. |
| HTTP 422 | Lees de foutmelding bij de aanvraag: gebruik geldige datums en een limiet van 1 tot 1000. |
| HTTP 429 | Je stuurde te veel aanvragen achter elkaar. Wacht even en gebruik de aangeleverde `ApiClient`. |
| HTTP 502 | Nginx kan de Python-dienst niet bereiken. Controleer de status van `iot-status`. |
| HTTP 503 | De verzameldatabase ontbreekt of is niet leesbaar. Controleer notebook 05 en de rechten op `/var/lib/iot`. |
| HTTP 409 of inhoudsverschil | De serverdatabase wijkt af van je eerdere controle. Controleer de server en wis geen bronbestanden om een fout te verbergen. |
| Lege bron bij import | Zorg dat in notebook 05 eerst beide volledige databases op je server zijn opgeslagen. |
| Verkeerde database gekozen | Gebruik het aparte bestand `analyse_06.db`, niet je brondatabase (`asserva.db` of `schauer.db`). |
| Onderbroken import | Herstel de verbinding en voer de importcel opnieuw uit. Een mislukte ronde wordt automatisch teruggedraaid. |
| NameError na hervatten | Voer de codecellen opnieuw in volgorde uit. Bestanden op schijf blijven staan, Python-variabelen in het geheugen niet. |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal)

Deze opdrachten tonen de dienstinstelling en recente logregels. Ze drukken geen opgeslagen wachtwoordbestand af.

</div>

```bash
sudo systemctl status iot-status --no-pager
sudo journalctl -u iot-status -n 30 --no-pager
sudo systemctl cat iot-status
```

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip</strong><br>Als de nieuwe API niet start, kun je tijdelijk terug naar de bereikbaarheidstest uit notebook 03. Hernoem alleen de aanvullende instelling en herstart de dienst met onderstaande opdrachten. De verzameldatabase blijft behouden. Herhaal daarna de installatie van deze uitbreiding zodra de oorzaak is opgelost.</div>

```bash
sudo mv /etc/systemd/system/iot-status.service.d/api-uitbreiding.conf /etc/systemd/system/iot-status.service.d/api-uitbreiding.conf.uit
sudo systemctl daemon-reload
sudo systemctl restart iot-status
```

## Technische bronnen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [FastAPI: queryparameters en validatie](https://fastapi.tiangolo.com/tutorial/query-params-str-validations/)
- [FastAPI: HTTP Basic-aanmelding](https://fastapi.tiangolo.com/advanced/security/http-basic-auth/)
- [Requests: sessies en certificaatcontrole](https://requests.readthedocs.io/en/latest/user/advanced/)
- [SQLite: database uitsluitend lezen met mode=ro](https://www.sqlite.org/uri.html)
- [Gegevenswoordenboek](../../gegevens/gegevenswoordenboek.md)
- [API-gegevenscontract](../../documentatie/api_gegevenscontract.md)

De technische documentatie is gecontroleerd op 25 november 2026.

</div>